# 1. Prompt Engineering — Standardisation

This notebook validates the **User Story Generation** prompt used by ClearSpec AI.

The objective is to ensure that stakeholder notes from different industries
(Healthcare, Finance, Retail, SaaS, etc.) are consistently transformed into
high-quality Agile User Stories.

The production prompt templates are defined in:

- `backend/prompts.py`
- `STORIES_SYSTEM`
- `stories_user_msg(...)`

Each generated response should satisfy the format checklist at the end of this notebook.

In [ ]:
import os
import sys
import time
import asyncio

sys.path.insert(0, "../backend")

from dotenv import load_dotenv

loaded = load_dotenv("../backend/.env")

if loaded:
    print("✓ Environment variables loaded.")
else:
    print("⚠ Warning: .env file not found.")

from llm_client import call_llm
from prompts import STORIES_SYSTEM, stories_user_msg


In [ ]:
SAMPLES = {

    "healthcare": """
Doctors want a faster way to see lab results.

Currently they log into three different systems.

Patients should receive either SMS or Email notifications when reports are available.

Critical lab values must immediately notify the on-call physician.

HIPAA-compliant audit logging is mandatory.
""",

    "finance": """
Traders need a dashboard displaying profit and loss across all books.

The dashboard should update in real time.

Compliance requires trades above $50,000 to be flagged.

The Risk team wants nightly stress-test simulations.
""",

    "retail": """
Store managers report that inventory searches are slow.

Employees should scan shelves to instantly identify out-of-stock items.

The system should recommend reorder quantities.

Loss-prevention wants alerts whenever expensive items leave the store unpaid.
""",

    "saas": """
Customers want role-based dashboards.

Administrators should manage user permissions.

Subscription renewals should happen automatically.

Users should receive reminders before subscriptions expire.
"""
}


In [ ]:
async def evaluate_prompt():

    print("=" * 80)
    print("CLEARSPEC AI - PROMPT VALIDATION")
    print("=" * 80)

    for domain, notes in SAMPLES.items():

        print()

        print("=" * 80)
        print(f"DOMAIN : {domain.upper()}")
        print("=" * 80)

        start = time.time()

        try:

            output = await call_llm(
                STORIES_SYSTEM,
                stories_user_msg(notes, domain)
            )

            elapsed = time.time() - start

            print(f"Response Time : {elapsed:.2f} sec")
            print(f"Characters    : {len(output)}")
            print("-" * 80)

            print(output[:1500])

            if len(output) > 1500:
                print("\n... output truncated ...")

        except Exception as e:

            print("❌ Generation failed")
            print(e)

    print()
    print("=" * 80)
    print("Prompt evaluation complete.")
    


In [ ]:
    # Run the prompt evaluation
    asyncio.run(evaluate_prompt())


# Output Evaluation Checklist

Verify every generated response satisfies the following.

## Structure

- [ ] Starts with `## User Stories`
- [ ] Stories are numbered correctly
- [ ] No hallucinated features
- [ ] Markdown formatting is consistent

## User Story Quality

Each story contains

- [ ] As a...
- [ ] I want...
- [ ] So that...

## Acceptance Criteria

- [ ] Given
- [ ] When
- [ ] Then

## Metadata

- [ ] Priority included
- [ ] Story estimate included
- [ ] Business value is clear

## Assumptions

- [ ] Ambiguous requirements marked with `[ASSUMED]`

## Domain Accuracy

- [ ] Healthcare terminology preserved
- [ ] Finance terminology preserved
- [ ] Retail terminology preserved
- [ ] SaaS terminology preserved

## Overall Score

Prompt Version: ___________

Format Accuracy: ____ / 10

Story Quality: ____ / 10

Production Ready:

- [ ] Yes
- [ ] Needs Improvement